<a href="https://colab.research.google.com/github/Gayuoff/AI-PoweredTypingTutor/blob/main/aiml_review_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import random
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
import matplotlib.pyplot as plt
from collections import Counter
import pandas as pd
import re

# Function to load words and sentences from a CSV file
def load_data_from_csv(filename):
    df = pd.read_csv(filename)
    words = df['Words'].tolist()
    sentences = df['Sentences'].tolist()
    return words, sentences

# Initialize mistyped words list and accuracy tracking
mistyped_words = []
accuracies = []
mistyped_letters_by_level = {1: [], 2: [], 3: [], 4: []}

# Finger mapping according to the universal format
finger_mapping = {
    '1': 'left pinky', '2': 'left ring finger', '3': 'left middle finger', '4': 'left index finger',
    '5': 'left index finger', '6': 'right index finger', '7': 'right index finger', '8': 'right middle finger',
    '9': 'right ring finger', '0': 'right pinky', '-': 'right pinky', '=': 'right pinky',
    'q': 'left pinky', 'w': 'left ring finger', 'e': 'left middle finger', 'r': 'left index finger',
    't': 'left index finger', 'y': 'right index finger', 'u': 'right index finger', 'i': 'right middle finger',
    'o': 'right ring finger', 'p': 'right pinky', '[': 'right pinky', ']': 'right pinky', '\\': 'right pinky',
    'a': 'left pinky', 's': 'left ring finger', 'd': 'left middle finger', 'f': 'left index finger',
    'g': 'left index finger', 'h': 'right index finger', 'j': 'right index finger', 'k': 'right middle finger',
    'l': 'right ring finger', ';': 'right pinky', "'": 'right pinky',
    'z': 'left pinky', 'x': 'left ring finger', 'c': 'left middle finger', 'v': 'left index finger',
    'b': 'left index finger', 'n': 'right index finger', 'm': 'right index finger', ',': 'right middle finger',
    '.': 'right ring finger', '/': 'right pinky', ' ': 'thumbs'
}

# Function to display finger instructions for typing a word
def display_finger_instructions(word):
    instructions = [finger_mapping.get(char, 'unknown') for char in word]
    print(f"Use the following fingers to type '{word}':")
    for char, instruction in zip(word, instructions):
        print(f"  {char.upper()}: {instruction}")

# Typing test function for words
def typing_test_words(items, level=1):
    sampled_items = random.sample(items, 5)
    correct_count = 0
    for item in sampled_items:
        display_finger_instructions(item)
        print(f'Type the following word: {item}')
        typed_item = input().lower()
        if typed_item != item:
            mistyped_words.append(typed_item)
            collect_mistyped_letters(typed_item, item, level)
        else:
            correct_count += 1
    typing_accuracy = correct_count / len(sampled_items) * 100
    accuracies.append(typing_accuracy)
    print(f"Typing accuracy for level {level}: {typing_accuracy}%")
    return typing_accuracy

# Typing test function for sentences
def typing_test_sentences(items, level):
    sampled_items = random.sample(items, 5)
    correct_count = 0
    for item in sampled_items:
        print(f'Type the following sentence: {item}')
        typed_item = input().lower()
        if typed_item != item:
            collect_mistyped_letters(typed_item, item, level)
        else:
            correct_count += 1
    typing_accuracy = correct_count / len(sampled_items) * 100
    accuracies.append(typing_accuracy)
    print(f"Typing accuracy for level {level}: {typing_accuracy}%")
    return typing_accuracy

# Function to collect mistyped letters
def collect_mistyped_letters(typed, original, level):
    for char_typed, char_original in zip(typed, original):
        if char_typed != char_original:
            mistyped_letters_by_level[level].append(char_typed)

# Visualize mistyped letters function
def visualize_mistyped_letters(level):
    letter_counts = Counter(mistyped_letters_by_level[level])
    all_letters = [chr(i) for i in range(97, 123)]
    counts = [letter_counts.get(letter, 0) for letter in all_letters]
    plt.figure(figsize=(10, 5))
    plt.bar(all_letters, counts, color='red')
    plt.xlabel('Letters')
    plt.ylabel('Mistyped Count')
    plt.title(f'Mistyped Letters Level {level}')
    plt.show()

# Function to find sentences with mistyped letters from the previous level
def get_sentences_with_mistyped_letters(sentences, mistyped_letters):
    sentences_with_mistyped = []
    for letter in mistyped_letters:
        sentences_with_mistyped.extend([sentence for sentence in sentences if letter in sentence])
    return list(set(sentences_with_mistyped))

# Function to find words with mistyped letters from the previous level
def get_words_with_mistyped_letters(words, mistyped_letters):
    words_with_mistyped = []
    for letter in mistyped_letters:
        words_with_mistyped.extend([word for word in words if letter in word])
    return list(set(words_with_mistyped))

# Main function
def main():
    # Load words and sentences from the CSV file
    words, sentences = load_data_from_csv('/content/words_sentences_meaningful - words_sentences_meaningful.csv.csv')

    # Level 1: Typing test with words
    typing_test_words(words, level=1)
    visualize_mistyped_letters(level=1)

    # Get the most mistyped letters from Level 1
    most_mistyped_letters_level1 = mistyped_letters_by_level[1]

    # Level 2: Typing test with 5 words that include the mistyped letters from Level 1
    if most_mistyped_letters_level1:
        level2_words = get_words_with_mistyped_letters(words, most_mistyped_letters_level1)
        typing_test_words(level2_words, level=2)
        visualize_mistyped_letters(level=2)
    else:
        print("No mistyped letters in Level 1 to generate Level 2 words.")

    # Level 3: Typing test with sentences
    typing_test_sentences(sentences, level=3)
    visualize_mistyped_letters(level=3)

    # Get the most mistyped letters from Level 3
    most_mistyped_letters_level3 = mistyped_letters_by_level[3]

    # Level 4: Typing test with sentences that include the mistyped letters from Level 3
    if most_mistyped_letters_level3:
        level4_sentences = get_sentences_with_mistyped_letters(sentences, most_mistyped_letters_level3)
        typing_test_sentences(level4_sentences, level=4)
        visualize_mistyped_letters(level=4)
    else:
        print("No mistyped letters in Level 3 to generate Level 4 sentences.")

    # Visualize typing accuracy across all levels
    plt.figure(figsize=(10, 5))
    plt.bar(['Level 1', 'Level 2', 'Level 3', 'Level 4'], accuracies, color=['blue', 'orange', 'green', 'purple'])
    plt.xlabel('Levels')
    plt.ylabel('Accuracy (%)')
    plt.title('Typing Accuracy by Level')
    plt.ylim(0, 100)
    plt.show()

    # Calculate overall accuracy
    overall_accuracy = sum(accuracies) / len(accuracies)
    print(f"Overall typing accuracy: {overall_accuracy}%")

if __name__ == '__main__':
    main()


Use the following fingers to type 'even':
  E: left middle finger
  V: left index finger
  E: left middle finger
  N: right index finger
Type the following word: even
